In [35]:
import json
from pathlib import Path

DATA_PATH = Path("scenario-foundry/data/generated_output/joensuu_messages.json")

with open(DATA_PATH, "r") as f:
    messages = json.load(f)

print(type(messages))

<class 'list'>


## 1.2 Inspect SAPIENT Message Structure

In [36]:
print("Total messages:", len(messages))
print("\nFirst message:")
print(json.dumps(messages[0], indent=2))

Total messages: 1234

First message:
{
  "sapientMessage": {
    "timestamp": "2026-11-15T02:45:00Z",
    "nodeId": "FI-MIL-RAD-KOLI-01",
    "detectionReport": {
      "objectId": "A-01-SWM-W1_DECOY",
      "state": "ACTIVE",
      "location": {
        "x": 30.631909,
        "y": 62.185001,
        "z": 1592.3,
        "coordinateSystem": "LOCATION_COORDINATE_SYSTEM_LAT_LNG_DEG_M",
        "datum": "LOCATION_DATUM_WGS84_E"
      },
      "objectInfo": [
        {
          "type": "estimatedSwarmCount",
          "value": "12"
        }
      ],
      "classification": [
        {
          "type": "Air vehicle",
          "confidence": 0.66,
          "subClass": [
            {
              "type": "UAV rotary wing",
              "level": 1,
              "subClass": [
                {
                  "type": "Military",
                  "level": 2,
                  "subClass": []
                }
              ]
            }
          ]
        }
      ],
      "trackInf

In [37]:
print("Top-level fields:")
print(messages[0].keys())

Top-level fields:
dict_keys(['sapientMessage'])


## 1.3 Dataset Overview

### inspect the whole dataset what kinds of messages

In [38]:
from collections import Counter

message_types = Counter()

for item in messages:
    msg = item["sapientMessage"]

    for key in msg:
        if key not in ["timestamp", "nodeId"]:
            message_types[key] += 1

print("Message types:")
for msg_type, count in message_types.items():
    print(f"{msg_type}: {count}")

Message types:
detectionReport: 1234


In [39]:
sensors = set()
objects = set()

for item in messages:
    msg = item["sapientMessage"]

    if "nodeId" in msg:
        sensors.add(msg["nodeId"])

    if "detectionReport" in msg:
        objects.add(msg["detectionReport"].get("objectId"))

print("Number of sensors:", len(sensors))
print("Sensors:", sensors)

print("\nNumber of detected objects:", len(objects))
print("Objects:", objects)

Number of sensors: 9
Sensors: {'FI-CIV-CAM-SAVONVOIMA-GATE', 'FI-CIV-ACU-SAVONVOIMA-01', 'FI-MIL-MDOP-RAIL-01', 'FI-MIL-MRAD-HEINA-02', 'FI-MIL-RAD-ONTTOLA-01', 'FI-BOR-EOIR-PEKKALABRIDGE-01', 'FI-MIL-MDOP-AIRPORT-WEST', 'FI-MIL-RAD-KOLI-01', 'FI-CIV-EOIR-SAVONVOIMA-STACK'}

Number of detected objects: 29
Objects: {'A-02-IND-W2_RAILWAY_08', 'CAM-A-01_W3_FPV', 'A-01-SWM-W2_PLANT', 'A-01-IND-W2_PLANT_08', 'ACU-A-01_W3_FPV', 'A-01-IND-W2_BRIDGE_08', 'CAM-A-01_W1_DECOY', 'ACU-A-01_W1_DECOY', 'ACU-A-01_W2_PLANT', 'A-01-IND-W2_RAILWAY_08', 'CAM-A-01_W2_BRIDGE', 'CAM-A-01_W2_RAILWAY', 'A-01-SWM-W3_FPV', 'A-01-IND-W1_DECOY_010', 'A-01-SWM-W2_RAILWAY', 'A-01-IND-W3_FPV_04', 'A-02-SWM-W2_AIRPORT', 'A-01-SWM-W2_BRIDGE', 'A-02-SWM-W2_BRIDGE', 'CAM-A-01_W2_PLANT', 'A-01-IND-W2_AIRPORT_08', 'ACU-A-01_W2_RAILWAY', 'A-02-SWM-W2_RAILWAY', 'A-02-SWM-W3_FPV', 'A-02-SWM-W2_PLANT', 'A-01-SWM-W2_AIRPORT', 'ACU-A-01_W2_BRIDGE', 'A-01-SWM-W1_DECOY', 'A-02-SWM-W1_DECOY'}


## 1.4 From Sensor and Object data

In [40]:
import pandas as pd

rows = []

for item in messages:
    msg = item["sapientMessage"]
    report = msg["detectionReport"]
    location = report.get("location", {})

    classification = report.get("classification", [])
    
    rows.append({
        "timestamp": msg.get("timestamp"),
        "sensor": msg.get("nodeId"),
        "object_id": report.get("objectId"),
        "state": report.get("state"),
        "longitude": location.get("x"),
        "latitude": location.get("y"),
        "altitude": location.get("z"),
        "classification": classification[0].get("type") if classification else None,
        "confidence": classification[0].get("confidence") if classification else None
    })


In [41]:
df = pd.DataFrame(rows)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="ISO8601",
    utc=True
)

print(df.shape)
df.head(10)

(1234, 9)


,timestamp,sensor,object_id,state,longitude,latitude,altitude,classification,confidence
0,2026-11-15 02:45:00+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.631909,62.185001,1592.3,Air vehicle,0.66
1,2026-11-15 02:45:17.996174+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.619391,62.191932,1593.4,Air vehicle,0.65
2,2026-11-15 02:45:35.989951+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.606374,62.197889,1594.7,Air vehicle,0.66
3,2026-11-15 02:45:54.024336+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.592777,62.205626,1596.2,Air vehicle,0.67
4,2026-11-15 02:46:11.954785+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.580902,62.212107,1597.8,Air vehicle,0.67
5,2026-11-15 02:46:11.954785+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_PLANT,ACTIVE,30.679781,62.154052,150.0,Air vehicle,0.61
6,2026-11-15 02:46:29.965752+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.568840,62.218747,1599.7,Air vehicle,0.66
7,2026-11-15 02:46:29.965752+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_PLANT,ACTIVE,30.664801,62.158378,150.2,Air vehicle,0.64
8,2026-11-15 02:46:47.965786+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,ACTIVE,30.554779,62.225571,1601.7,Air vehicle,0.63
9,2026-11-15 02:46:47.965786+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_PLANT,ACTIVE,30.651226,62.163234,150.6,Air vehicle,0.59


### summarize each object

In [42]:
object_summary = (
    df.groupby("object_id")
    .agg(
        detections=("object_id", "size"),
        sensors=("sensor", "nunique"),
        min_altitude=("altitude", "min"),
        max_altitude=("altitude", "max"),
        mean_altitude=("altitude", "mean"),
        min_confidence=("confidence", "min"),
        max_confidence=("confidence", "max"),
        mean_confidence=("confidence", "mean"),
    )
    .round(2)
    .sort_values("detections", ascending=False)
)

object_summary

,detections,sensors,min_altitude,max_altitude,mean_altitude,min_confidence,max_confidence,mean_confidence
object_id,,,,,,,,
A-01-SWM-W2_AIRPORT,149,2,169.4,225.0,206.82,0.44,0.89,0.75
A-01-SWM-W2_RAILWAY,130,2,143.6,247.1,205.23,0.45,0.84,0.71
A-01-SWM-W2_BRIDGE,122,2,145.5,203.4,178.24,0.44,0.84,0.69
A-01-SWM-W2_PLANT,120,2,148.1,214.0,186.11,0.43,0.84,0.68
A-01-SWM-W1_DECOY,108,2,1584.3,1664.3,1629.19,0.47,0.83,0.69
A-02-SWM-W2_BRIDGE,73,1,145.6,203.4,182.70,0.44,0.71,0.62
A-02-SWM-W1_DECOY,68,1,1583.7,1664.3,1635.29,0.46,0.82,0.67
A-02-SWM-W2_PLANT,64,1,148.1,214.0,193.24,0.45,0.79,0.67
A-02-SWM-W2_RAILWAY,60,1,144.3,239.4,198.96,0.43,0.80,0.64


In [43]:
print("Time range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())

print("\nAltitude range:")
print(df["altitude"].min(), "to", df["altitude"].max())

print("\nConfidence range:")
print(df["confidence"].min(), "to", df["confidence"].max())

Time range:
2026-11-15 02:45:00+00:00 to 2026-11-15 03:16:24.998677+00:00

Altitude range:
103.1 to 1664.3

Confidence range:
0.12 to 0.97


### find the strongest pattern

In [44]:
sensor_summary = (
    df.groupby("sensor")
    .agg(
        detections=("sensor", "size"),
        objects=("object_id", "nunique"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min"),
        max_confidence=("confidence", "max"),
        location_available=("altitude", "count")
    )
    .round(2)
    .sort_values("mean_confidence")
)

sensor_summary

,detections,objects,mean_confidence,min_confidence,max_confidence,location_available
sensor,,,,,,
FI-CIV-CAM-SAVONVOIMA-GATE,34,5,0.12,0.12,0.12,34
FI-BOR-EOIR-PEKKALABRIDGE-01,34,5,0.43,0.28,0.97,34
FI-CIV-EOIR-SAVONVOIMA-STACK,44,5,0.48,0.27,0.94,44
FI-CIV-ACU-SAVONVOIMA-01,53,5,0.54,0.33,0.85,0
FI-MIL-MRAD-HEINA-02,354,7,0.64,0.43,0.86,354
FI-MIL-RAD-ONTTOLA-01,280,7,0.66,0.43,0.95,280
FI-MIL-MDOP-RAIL-01,31,5,0.67,0.50,0.92,31
FI-MIL-MDOP-AIRPORT-WEST,6,1,0.71,0.48,0.91,6
FI-MIL-RAD-KOLI-01,398,6,0.75,0.59,0.89,398


In [45]:
df.nsmallest(15, "confidence")[
    ["timestamp", "sensor", "object_id", "altitude", "confidence"]
]

,timestamp,sensor,object_id,altitude,confidence
292,2026-11-15 02:57:21.987143+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,106.6,0.12
304,2026-11-15 02:57:36.016470+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,105.8,0.12
330,2026-11-15 02:57:49.992292+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,105.1,0.12
348,2026-11-15 02:58:04.013061+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,104.5,0.12
366,2026-11-15 02:58:17.990689+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,104.0,0.12
387,2026-11-15 02:58:32.001954+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,103.6,0.12
405,2026-11-15 02:58:46.026963+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,103.3,0.12
426,2026-11-15 02:58:59.989717+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,103.1,0.12
450,2026-11-15 02:59:14.014722+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W3_FPV,103.1,0.12
657,2026-11-15 03:02:43.957949+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,1589.6,0.12


Pattern 1 One camera has consistently very low confidence. FI-CIV-CAM-SAVONVOIMA-GATE  34 detections across 5 objects, and every single detection has confidence 0.12.

### Findings in altitude profiles

In [46]:
altitude_objects = (
    df[df["altitude"].notna()]
    .groupby("object_id")
    .agg(
        detections=("altitude", "count"),
        min_altitude=("altitude", "min"),
        max_altitude=("altitude", "max"),
        mean_altitude=("altitude", "mean")
    )
    .round(1)
    .sort_values("mean_altitude")
)

altitude_objects

,detections,min_altitude,max_altitude,mean_altitude
object_id,,,,
A-01-IND-W3_FPV_04,7,103.1,103.9,103.4
CAM-A-01_W3_FPV,33,103.1,109.4,104.9
A-01-SWM-W3_FPV,35,103.1,116.4,108.2
A-02-SWM-W3_FPV,23,103.1,117.0,108.5
A-01-IND-W2_RAILWAY_08,8,143.6,149.5,147.1
A-01-IND-W2_BRIDGE_08,8,145.6,148.1,147.2
CAM-A-01_W2_BRIDGE,23,145.5,150.9,147.6
A-01-IND-W2_PLANT_08,1,148.1,148.1,148.1
CAM-A-01_W2_RAILWAY,24,143.6,155.4,148.4


## 1.5 Finding Sudden Altitude Changes

changes within the same object

In [47]:
altitude_change = (
    df[df["altitude"].notna()]
    .sort_values(["object_id", "timestamp"])
    .copy()
)

altitude_change["altitude_diff"] = (
    altitude_change
    .groupby("object_id")["altitude"]
    .diff()
    .abs()
)

altitude_change.nlargest(15, "altitude_diff")[
    ["timestamp", "sensor", "object_id", "altitude", "altitude_diff"]
]

,timestamp,sensor,object_id,altitude,altitude_diff
855,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,208.0,31.4
859,2026-11-15 03:05:35.958316+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,206.2,5.4
847,2026-11-15 03:05:24.033336+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,211.6,5.3
879,2026-11-15 03:05:59.970874+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,195.5,5.3
891,2026-11-15 03:06:12.014886+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,190.4,5.1
867,2026-11-15 03:05:42.981555+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,203.0,5.0
875,2026-11-15 03:05:54.031189+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,198.1,4.9
827,2026-11-15 03:05:00.007841+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,222.1,4.8
843,2026-11-15 03:05:20.973326+00:00,FI-MIL-MRAD-HEINA-02,A-02-IND-W2_RAILWAY_08,213.0,4.8
887,2026-11-15 03:06:04.983205+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,193.3,4.8


In [48]:
altitude_change = (
    df[df["altitude"].notna()]
    .sort_values(["object_id", "sensor", "timestamp"])
    .copy()
)

altitude_change["altitude_diff"] = (
    altitude_change
    .groupby(["object_id", "sensor"])["altitude"]
    .diff()
    .abs()
)

largest_altitude_changes = (
    altitude_change
    .nlargest(15, "altitude_diff")
    [["timestamp", "sensor", "object_id", "altitude", "altitude_diff"]]
)

largest_altitude_changes

,timestamp,sensor,object_id,altitude,altitude_diff
855,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,208.0,31.4
863,2026-11-15 03:05:41.965886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,203.5,8.1
851,2026-11-15 03:05:24.044954+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,211.6,8.0
879,2026-11-15 03:05:59.970874+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,195.5,8.0
900,2026-11-15 03:06:18.043691+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,187.9,7.6
831,2026-11-15 03:05:05.968450+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,219.6,7.3
508,2026-11-15 03:00:18.046300+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,1629.0,7.1
525,2026-11-15 03:00:35.974159+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,1622.0,7.0
918,2026-11-15 03:06:35.975965+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,180.9,7.0
542,2026-11-15 03:00:54.044970+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,1615.1,6.9


## 1.6 Finding the Largest Altitude Change

The largest observed altitude change is 31.4 m for `A-02-SWM-W2_RAILWAY`, detected by `FI-MIL-MRAD-HEINA-02`. Since this change is much larger than the other consecutive changes, the surrounding detections are inspected to determine whether it represents an isolated jump or part of a continuous movement pattern.

In [49]:
target = df[
    (df["object_id"] == "A-02-SWM-W2_RAILWAY") &
    (df["sensor"] == "FI-MIL-MRAD-HEINA-02")
][
    ["timestamp", "sensor", "object_id", "altitude", "confidence"]
].sort_values("timestamp")

target

,timestamp,sensor,object_id,altitude,confidence
179,2026-11-15 02:55:27.029027+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,212.9,0.43
191,2026-11-15 02:55:38.022630+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,213.4,0.46
203,2026-11-15 02:55:48.972300+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,213.9,0.50
207,2026-11-15 02:55:59.955698+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,214.5,0.48
219,2026-11-15 02:56:11.031829+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,215.0,0.52
225,2026-11-15 02:56:21.990872+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,215.5,0.49
238,2026-11-15 02:56:33.007809+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,216.0,0.53
252,2026-11-15 02:56:43.970095+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,216.6,0.53
260,2026-11-15 02:56:55.033181+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,217.1,0.54
274,2026-11-15 02:57:05.969434+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,217.7,0.54


## 1.7 Investigating Detection Gaps

A large altitude difference can be misleading when there is a long time gap between detections. To identify possible interruptions in sensor observations, the time interval between consecutive detections is calculated for each object and sensor.

In [50]:
detection_gaps = (
    df.sort_values(["object_id", "sensor", "timestamp"])
    .copy()
)

detection_gaps["time_gap_seconds"] = (
    detection_gaps
    .groupby(["object_id", "sensor"])["timestamp"]
    .diff()
    .dt.total_seconds()
)

largest_gaps = detection_gaps.nlargest(
    15, "time_gap_seconds"
)[
    [
        "timestamp",
        "sensor",
        "object_id",
        "altitude",
        "confidence",
        "time_gap_seconds"
    ]
]

largest_gaps

,timestamp,sensor,object_id,altitude,confidence,time_gap_seconds
855,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,208.0,0.77,241.976227
785,2026-11-15 03:04:12.041895+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_AIRPORT,224.2,0.83,18.091337
784,2026-11-15 03:04:12.041895+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_BRIDGE,169.3,0.77,18.091337
782,2026-11-15 03:04:12.041895+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_PLANT,203.6,0.80,18.091337
783,2026-11-15 03:04:12.041895+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,238.7,0.78,18.091337
64,2026-11-15 02:50:24.045886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W1_DECOY,1628.3,0.69,18.085253
68,2026-11-15 02:50:24.045886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_AIRPORT,213.0,0.73,18.085253
67,2026-11-15 02:50:24.045886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_BRIDGE,171.2,0.64,18.085253
65,2026-11-15 02:50:24.045886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_PLANT,166.7,0.68,18.085253
66,2026-11-15 02:50:24.045886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,200.7,0.71,18.085253


## 1.8 Interesting Patterns Observed

The generated Joensuu scenario contains 1,234 SAPIENT detection reports from 9 sensors covering 29 object IDs. Two patterns stood out during inspection:

1. **Consistently low camera confidence:** `FI-CIV-CAM-SAVONVOIMA-GATE` produced 34 detections across 5 objects, but every detection had a confidence score of only `0.12`. This is noticeably lower than the other sensors, whose average confidence ranged from approximately `0.43` to `0.75`. This suggests a sensor-specific low-confidence pattern.

2. **Long detection gap for a railway target:** `A-02-SWM-W2_RAILWAY`, observed by `FI-MIL-MRAD-HEINA-02`, had a detection gap of approximately `242 seconds`. Most of the other large gaps were only around `18 seconds`. The target was later detected again at an altitude of `208 m`, after previously being observed at `239.4 m`. This suggests a temporary loss of observation followed by target reacquisition.

# Task 2: Simple Anomaly Detection

A lightweight rule-based detector is used to identify noteworthy SAPIENT detections. The detector focuses on interpretable signals observed during initial data exploration: unusually low detection confidence, long gaps between consecutive observations, and sudden altitude changes. Each flagged message includes the reason it was identified, making the results easy to inspect and explain.

## 2.1 Calculate Temporal Features

For each object observed by the same sensor, the time gap and altitude change between consecutive detections are calculated. Altitude change is converted to a rate so that a large change over several minutes is not treated the same as a large change over a few seconds.

In [51]:
anomaly_df = df.sort_values(
    ["object_id", "sensor", "timestamp"]
).copy()

# Time since previous detection
anomaly_df["time_gap_seconds"] = (
    anomaly_df
    .groupby(["object_id", "sensor"])["timestamp"]
    .diff()
    .dt.total_seconds()
)

# Altitude difference from previous detection
anomaly_df["altitude_change"] = (
    anomaly_df
    .groupby(["object_id", "sensor"])["altitude"]
    .diff()
    .abs()
)

# Altitude change per second
anomaly_df["altitude_rate"] = (
    anomaly_df["altitude_change"] /
    anomaly_df["time_gap_seconds"]
)

anomaly_df[
    [
        "timestamp",
        "sensor",
        "object_id",
        "confidence",
        "time_gap_seconds",
        "altitude_change",
        "altitude_rate"
    ]
].describe()

,confidence,time_gap_seconds,altitude_change,altitude_rate
count,1234.000000,1188.000000,1140.000000,1140.000000
mean,0.651305,13.813996,1.421754,0.103515
std,0.158345,7.364852,1.583159,0.095892
min,0.120000,8.922142,0.000000,0.000000
25%,0.570000,11.019496,0.600000,0.044432
50%,0.680000,12.001462,1.000000,0.075157
75%,0.770000,17.970015,1.800000,0.118853
max,0.970000,241.976227,31.400000,0.453267


## 2.2 Define Statistical Anomaly Thresholds

In [52]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return lower, upper


confidence_lower, _ = iqr_bounds(anomaly_df["confidence"])
_, gap_upper = iqr_bounds(anomaly_df["time_gap_seconds"].dropna())
_, altitude_rate_upper = iqr_bounds(anomaly_df["altitude_rate"].dropna())

print(f"Low confidence threshold: {confidence_lower:.3f}")
print(f"Long detection gap threshold: {gap_upper:.3f} seconds")
print(f"High altitude rate threshold: {altitude_rate_upper:.3f} m/s")

Low confidence threshold: 0.270
Long detection gap threshold: 28.396 seconds
High altitude rate threshold: 0.230 m/s


## 2.3 Flag Anomalous Messages

In [54]:
def detect_anomalies(row):
    reasons = []

    if row["confidence"] < confidence_lower:
        reasons.append("Low confidence")

    if pd.notna(row["time_gap_seconds"]) and row["time_gap_seconds"] > gap_upper:
        reasons.append("Long detection gap")

    if pd.notna(row["altitude_rate"]) and row["altitude_rate"] > altitude_rate_upper:
        reasons.append("High altitude change rate")

    return ", ".join(reasons)


anomaly_df["anomaly_reason"] = anomaly_df.apply(
    detect_anomalies,
    axis=1
)

anomaly_df["is_anomaly"] = anomaly_df["anomaly_reason"] != ""

detected = anomaly_df[anomaly_df["is_anomaly"]]

print("Total messages:", len(anomaly_df))
print("Flagged messages:", len(detected))

print("\nFlags by reason:")
print(detected["anomaly_reason"].value_counts())

Total messages: 1234
Flagged messages: 159

Flags by reason:
anomaly_reason
High altitude change rate    124
Low confidence                34
Long detection gap             1
Name: count, dtype: int64


## 2.4 Inspect Detected Anomalies

The flagged messages are inspected to verify whether the statistical rules identify meaningful patterns. Examples from each anomaly type are examined before interpreting the results.

inspect the highest altitude rates

In [55]:
altitude_anomalies = detected[
    detected["anomaly_reason"] == "High altitude change rate"
].sort_values("altitude_rate", ascending=False)

altitude_anomalies[
    [
        "timestamp",
        "sensor",
        "object_id",
        "altitude",
        "time_gap_seconds",
        "altitude_change",
        "altitude_rate",
        "confidence"
    ]
].head(10)

,timestamp,sensor,object_id,altitude,time_gap_seconds,altitude_change,altitude_rate,confidence
867,2026-11-15 03:05:42.981555+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,203.0,11.031036,5.0,0.453267,0.76
859,2026-11-15 03:05:35.958316+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,206.2,11.924980,5.4,0.452831,0.68
863,2026-11-15 03:05:41.965886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,203.5,17.920932,8.1,0.451985,0.82
871,2026-11-15 03:05:48.023141+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,200.8,12.064825,5.4,0.447582,0.67
879,2026-11-15 03:05:59.970874+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,195.5,18.004988,8.0,0.444321,0.82
875,2026-11-15 03:05:54.031189+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,198.1,11.049634,4.9,0.443454,0.78
851,2026-11-15 03:05:24.044954+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_RAILWAY,211.6,18.076504,8.0,0.442563,0.80
883,2026-11-15 03:06:00.021491+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,195.5,11.998350,5.3,0.441727,0.71
847,2026-11-15 03:05:24.033336+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,211.6,12.028221,5.3,0.440630,0.65
843,2026-11-15 03:05:20.973326+00:00,FI-MIL-MRAD-HEINA-02,A-02-IND-W2_RAILWAY_08,213.0,10.943943,4.8,0.438599,0.78


inspect the low-confidence

In [56]:
low_confidence_anomalies = detected[
    detected["anomaly_reason"] == "Low confidence"
]

low_confidence_anomalies[
    [
        "timestamp",
        "sensor",
        "object_id",
        "confidence"
    ]
].head(10)

,timestamp,sensor,object_id,confidence
657,2026-11-15 03:02:43.957949+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
672,2026-11-15 03:02:58.029652+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
696,2026-11-15 03:03:12.045877+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
714,2026-11-15 03:03:25.964406+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
736,2026-11-15 03:03:40.040238+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
762,2026-11-15 03:03:54.041422+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
776,2026-11-15 03:04:07.950609+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12
926,2026-11-15 03:06:42.019230+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W2_BRIDGE,0.12
941,2026-11-15 03:06:55.983005+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W2_BRIDGE,0.12
957,2026-11-15 03:07:10.041429+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W2_BRIDGE,0.12


long-gap

In [57]:
gap_anomalies = detected[
    detected["anomaly_reason"] == "Long detection gap"
]

gap_anomalies[
    [
        "timestamp",
        "sensor",
        "object_id",
        "time_gap_seconds",
        "altitude",
        "confidence"
    ]
]

,timestamp,sensor,object_id,time_gap_seconds,altitude,confidence
855,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,241.976227,208.0,0.77


## 2.5 Show Example Detections

In [58]:
examples = pd.concat([
    detected[detected["anomaly_reason"] == "Low confidence"].head(1),
    detected[detected["anomaly_reason"] == "Long detection gap"].head(1),
    detected[
        detected["anomaly_reason"] == "High altitude change rate"
    ].sort_values("altitude_rate", ascending=False).head(1)
])

examples[
    [
        "timestamp",
        "sensor",
        "object_id",
        "confidence",
        "time_gap_seconds",
        "altitude_change",
        "altitude_rate",
        "anomaly_reason"
    ]
]

,timestamp,sensor,object_id,confidence,time_gap_seconds,altitude_change,altitude_rate,anomaly_reason
657,2026-11-15 03:02:43.957949+00:00,FI-CIV-CAM-SAVONVOIMA-GATE,CAM-A-01_W1_DECOY,0.12,NaN,NaN,NaN,Low confidence
855,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,0.77,241.976227,31.4,0.129765,Long detection gap
867,2026-11-15 03:05:42.981555+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,0.76,11.031036,5.0,0.453267,High altitude change rate


## 2.6 Results

The lightweight IQR-based detector flagged 159 of 1,234 messages as noteworthy. 
The IQR (Interquartile Range) represents the range between the 25th percentile (Q1) and 75th percentile (Q3). Values outside 1.5 × IQR from this range are treated as noteworthy. This method automatically calculated thresholds for confidence, detection gaps, and altitude change rates from the generated data.

Three example detections illustrate the different types of flags:

1. **Low confidence:** `CAM-A-01_W1_DECOY` was detected with a confidence of `0.12`, below the calculated threshold of `0.27`.

2. **Long detection gap:** `A-02-SWM-W2_RAILWAY` was detected after a gap of approximately `242 seconds`, well above the calculated threshold of `28.4 seconds`.

3. **High altitude change rate:** `A-02-SWM-W2_RAILWAY` changed altitude by `5.0 m` over approximately `11 seconds`, corresponding to `0.45 m/s`, above the calculated threshold of `0.23 m/s`.

The detector is intentionally lightweight and interpretable. Each flag is based on a simple statistical threshold, and the reason for flagging is retained with the message. The flagged observations should be treated as noteworthy events rather than necessarily sensor errors.

# Task 3: LLM Situation Summary

Small Batch

In [59]:
start_time = pd.Timestamp("2026-11-15 03:05:20", tz="UTC")
end_time = pd.Timestamp("2026-11-15 03:06:00", tz="UTC")

llm_batch = anomaly_df[
    (anomaly_df["timestamp"] >= start_time) &
    (anomaly_df["timestamp"] <= end_time)
].copy()

print("Messages selected:", len(llm_batch))

llm_batch[
    [
        "timestamp",
        "sensor",
        "object_id",
        "altitude",
        "confidence"
    ]
]

Messages selected: 40


,timestamp,sensor,object_id,altitude,confidence
853,2026-11-15 03:05:24.044954+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_AIRPORT,221.7,0.81
865,2026-11-15 03:05:41.965886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_AIRPORT,220.8,0.86
881,2026-11-15 03:05:59.970874+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_AIRPORT,219.8,0.84
849,2026-11-15 03:05:24.033336+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_AIRPORT,221.7,0.52
861,2026-11-15 03:05:35.958316+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_AIRPORT,221.1,0.53
873,2026-11-15 03:05:48.023141+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_AIRPORT,220.5,0.54
852,2026-11-15 03:05:24.044954+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_BRIDGE,157.0,0.78
864,2026-11-15 03:05:41.965886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_BRIDGE,154.5,0.83
880,2026-11-15 03:05:59.970874+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_BRIDGE,152.4,0.82
848,2026-11-15 03:05:24.033336+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_BRIDGE,157.0,0.71


## 3.1 Prepare a Small LLM Input Batch

by taking 12 points

In [61]:
llm_input = (
    llm_batch
    .sort_values("timestamp")
    .iloc[::3]
    .head(12)
    [
        [
            "timestamp",
            "sensor",
            "object_id",
            "altitude",
            "confidence"
        ]
    ]
)

llm_input

,timestamp,sensor,object_id,altitude,confidence
843,2026-11-15 03:05:20.973326+00:00,FI-MIL-MRAD-HEINA-02,A-02-IND-W2_RAILWAY_08,213.0,0.78
842,2026-11-15 03:05:20.973326+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_PLANT,180.0,0.73
847,2026-11-15 03:05:24.033336+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_RAILWAY,211.6,0.65
850,2026-11-15 03:05:24.044954+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_PLANT,178.9,0.79
856,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_BRIDGE,155.9,0.63
855,2026-11-15 03:05:31.950519+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,208.0,0.77
858,2026-11-15 03:05:35.958316+00:00,FI-MIL-RAD-ONTTOLA-01,A-01-SWM-W2_PLANT,174.9,0.71
864,2026-11-15 03:05:41.965886+00:00,FI-MIL-RAD-KOLI-01,A-01-SWM-W2_BRIDGE,154.5,0.83
866,2026-11-15 03:05:42.981555+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_PLANT,172.6,0.73
867,2026-11-15 03:05:42.981555+00:00,FI-MIL-MRAD-HEINA-02,A-02-SWM-W2_RAILWAY,203.0,0.76


In [62]:
detection_text = llm_input.to_string(index=False)

print(detection_text)

                       timestamp                sensor              object_id  altitude  confidence
2026-11-15 03:05:20.973326+00:00  FI-MIL-MRAD-HEINA-02 A-02-IND-W2_RAILWAY_08     213.0        0.78
2026-11-15 03:05:20.973326+00:00  FI-MIL-MRAD-HEINA-02      A-02-SWM-W2_PLANT     180.0        0.73
2026-11-15 03:05:24.033336+00:00 FI-MIL-RAD-ONTTOLA-01    A-01-SWM-W2_RAILWAY     211.6        0.65
2026-11-15 03:05:24.044954+00:00    FI-MIL-RAD-KOLI-01      A-01-SWM-W2_PLANT     178.9        0.79
2026-11-15 03:05:31.950519+00:00  FI-MIL-MRAD-HEINA-02     A-02-SWM-W2_BRIDGE     155.9        0.63
2026-11-15 03:05:31.950519+00:00  FI-MIL-MRAD-HEINA-02    A-02-SWM-W2_RAILWAY     208.0        0.77
2026-11-15 03:05:35.958316+00:00 FI-MIL-RAD-ONTTOLA-01      A-01-SWM-W2_PLANT     174.9        0.71
2026-11-15 03:05:41.965886+00:00    FI-MIL-RAD-KOLI-01     A-01-SWM-W2_BRIDGE     154.5        0.83
2026-11-15 03:05:42.981555+00:00  FI-MIL-MRAD-HEINA-02      A-02-SWM-W2_PLANT     172.6        0.73


## 3.2 LLM Part

In [63]:
prompt = f"""
You are assisting a human operator monitoring sensor detections.

Using only the detections provided below, write a short situation summary
of 2-4 sentences.

Focus on:
- which objects are being detected,
- where relevant, how their altitude changes,
- confidence of the observations,
- any noteworthy pattern visible in the data.

Do not invent object types, locations, intentions, threats, or events that
are not explicitly supported by the detections.

Detections:
{detection_text}

Situation summary:
"""

print(prompt)


You are assisting a human operator monitoring sensor detections.

Using only the detections provided below, write a short situation summary
of 2-4 sentences.

Focus on:
- which objects are being detected,
- where relevant, how their altitude changes,
- confidence of the observations,
- any noteworthy pattern visible in the data.

Do not invent object types, locations, intentions, threats, or events that
are not explicitly supported by the detections.

Detections:
                       timestamp                sensor              object_id  altitude  confidence
2026-11-15 03:05:20.973326+00:00  FI-MIL-MRAD-HEINA-02 A-02-IND-W2_RAILWAY_08     213.0        0.78
2026-11-15 03:05:20.973326+00:00  FI-MIL-MRAD-HEINA-02      A-02-SWM-W2_PLANT     180.0        0.73
2026-11-15 03:05:24.033336+00:00 FI-MIL-RAD-ONTTOLA-01    A-01-SWM-W2_RAILWAY     211.6        0.65
2026-11-15 03:05:24.044954+00:00    FI-MIL-RAD-KOLI-01      A-01-SWM-W2_PLANT     178.9        0.79
2026-11-15 03:05:31.950519+00:0

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"

print("Key set")

Key set


In [76]:
api_key = os.getenv("GEMINI_API_KEY")
print(api_key is not None)

True


## 3.4 Generate Situation Summary with Gemini

The selected detections are provided to Gemini, an LLM, with instructions to generate a short operator-oriented situation summary. The prompt asks the model to use only the provided sensor information and avoid unsupported assumptions.

In [78]:
from google import genai
import os

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

summary = response.text

print(summary)

Between 03:05:20 and 03:05:54 UTC, three sensors recorded twelve detections corresponding to seven unique object IDs (designations under the A-01 and A-02 series referencing RAILWAY, PLANT, and BRIDGE targets). Observation confidence levels across all detections range between 0.63 and 0.83, with overall recorded altitudes ranging from 213.0 down to 153.0. A consistent pattern across every object detected multiple times (such as A-01-SWM-W2_PLANT, A-02-SWM-W2_PLANT, A-02-SWM-W2_RAILWAY, and A-02-SWM-W2_BRIDGE) is a steady decrease in altitude over successive timestamps.


Gemini Response : Between 03:05:20 and 03:05:54 UTC, three sensors recorded twelve detections corresponding to seven unique object IDs (designations under the A-01 and A-02 series referencing RAILWAY, PLANT, and BRIDGE targets). Observation confidence levels across all detections range between 0.63 and 0.83, with overall recorded altitudes ranging from 213.0 down to 153.0. A consistent pattern across every object detected multiple times (such as A-01-SWM-W2_PLANT, A-02-SWM-W2_PLANT, A-02-SWM-W2_RAILWAY, and A-02-SWM-W2_BRIDGE) is a steady decrease in altitude over successive timestamps.

## 3.5 Critical Evaluation of the LLM Summary

The generated summary was generally accurate and useful for a human operator. It correctly identified the detected object groups, observation time range, confidence range, and altitude range.

**Good aspects:**
- The LLM correctly summarized the observation period from `03:05:20` to `03:05:54 UTC`.
- It correctly reported that multiple sensors were involved, including `FI-MIL-MRAD-HEINA-02`, `FI-MIL-RAD-ONTTOLA-01`, and `FI-MIL-RAD-KOLI-01`.
- It correctly identified the confidence range (`0.63` to `0.83`) and altitude range (`153.0 m` to `213.0 m`) from the provided detections.
- It avoided unsupported claims about threats, intentions, or object types.

**Limitations and possible hallucination:**
- The statement that "every object detected multiple times shows a steady decrease in altitude" is slightly too strong. For example, `A-02-SWM-W2_RAILWAY` decreases from `208.0 m` to `203.0 m`, and `A-02-SWM-W2_PLANT` decreases from `180.0 m` to `172.6 m`, but the sample only contains a short observation window and does not prove this trend for all objects.
- The LLM referred to the detections as "targets". The dataset only provides `object_id` values and sensor measurements, so "target" is an interpretation rather than explicit information.
- No major hallucinations were observed, but the summary shows that LLM-generated situation reports still require verification against the original sensor data before operational use.

# Task 4 

### Reflection
AI tools were most helpful during the development process by saving time when exploring the dataset, debugging code, and understanding the structure of the SAPIENT messages. They helped me understand the generated sensor data, design a simple anomaly detection method, and convert multiple sensor detections into a short situation summary for a human operator. The LLM was useful for turning raw sensor information into a more readable summary that is easier to understand.


However, AI suggestions were not always fully correct and needed to be checked against the original data. Some suggestions were too general and did not always consider the details of the dataset. For example, the LLM summary correctly identified altitude changes and confidence patterns, but it made the conclusion that every repeatedly detected object showed decreasing altitude. In reality, only some objects showed this pattern within the selected time period. The LLM also used the word "targets", although the data only contained object IDs and sensor measurements. This shows that LLM outputs should always be reviewed before being used for decision-making.


If this system were used in production, the biggest challenges would be reliability, false alarms, and missing context. The current anomaly detection method is simple and can identify unusual patterns, but it may also flag normal situations as anomalies. A real system would need more historical data, better tracking methods, domain knowledge, and validation from human operators. The LLM summary would also need strong connection to verified sensor data to reduce incorrect information. Overall, AI tools improved the development process, but human review is still important, especially for systems used in safety-critical situations.


Overall, AI tools helped throughout the project by improving data understanding, supporting the implementation of anomaly detection, and generating a useful situation summary. At the same time, the results showed that AI outputs need careful checking, and a production system would require stronger validation, better context handling, and human supervision.
